In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib captum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 17.8 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
from transformers import TrainingArguments, Trainer
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset, Audio, DatasetDict
from sklearn.model_selection import GroupShuffleSplit

from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

###Ορισμός dictionaries για μοντέλα και dataset

In [ ]:
MODEL_REPOS = {
    "TESS":    "VasilisAsim/data2vec-finetuned-TESS_full",
    "RAVDESS": "VasilisAsim/data2vec-finetuned-ravdess",
    "CREMA-D": "VasilisAsim/data2vec-finetuned-CREMA-D",
    "IEMOCAP": "VasilisAsim/data2vec-finetuned-IEMOCAP-categorical"
}

DATASET_REPOS = {
    "TESS":    "AbstractTTS/TESS",
    "RAVDESS": "narad/ravdess",
    "CREMA-D": "AbstractTTS/CREMA-D",
    "IEMOCAP": "AbstractTTS/IEMOCAP",
}

###Φόρτωση όλων των μοντέλων και feature extractors σε dictionaries

In [ ]:
#Φόρτωση όλων των μοντέλων και feature extractors σε dictionaries
models = {}
feature_extractors = {}

for corpus, repo_id in MODEL_REPOS.items():
    print(f"Φόρτωση μοντέλου: {corpus} ({repo_id})")
    feature_extractors[corpus] = AutoFeatureExtractor.from_pretrained(repo_id)
    models[corpus] = AutoModelForAudioClassification.from_pretrained(repo_id)
    models[corpus].eval()
    if torch.cuda.is_available():
        models[corpus] = models[corpus].to("cuda")

Φόρτωση μοντέλου: TESS (VasilisAsim/data2vec-finetuned-TESS_full)


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  373MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/234 [00:00<?, ?it/s]

... (14 outputs left out)

config.json:   0%|          | 0.00/1.85k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  373MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/234 [00:00<?, ?it/s]

### Συναρτήσεις και φόρτωση για όλα τα dataset

In [ ]:
def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))
    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename: return "OAF"
        elif "YAF" in filename: return "YAF"
        return "unknown"
    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")
    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    return DatasetDict({"train": train_dataset, "test": test_dataset})


def load_ravdess_speaker_independent():
    dataset = load_dataset("narad/ravdess", revision="refs/convert/parquet")["train"]
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

    splitter = GroupShuffleSplit(test_size=0.35, n_splits=1, random_state=42)  # ⚠️ επιβεβαίωσε 0.35
    train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=dataset["speaker_id"]))
    dataset = DatasetDict({
        "train": dataset.select(train_idx),
        "test": dataset.select(test_idx)
    })

    # label_names χρειάζονται για το label_mapping.py (int→string)
    dataset_raw = load_dataset("narad/ravdess", revision="refs/convert/parquet")["train"]
    label_names = dataset_raw.features["labels"].int2str(list(range(dataset_raw.features["labels"].num_classes)))
    dataset.ravdess_label_names = label_names  # βολικό attribute για μετά, όχι απαραίτητο
    return dataset


def load_cremad_speaker_independent():
    dataset = load_dataset("AbstractTTS/CREMA-D", split="train")
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))
    def add_speaker(batch):
        batch["actor_id"] = batch["file"].split("_")[0]
        return batch
    dataset = dataset.map(add_speaker)
    splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
    train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=dataset["actor_id"]))
    return DatasetDict({
        "train": dataset.select(train_idx),
        "test": dataset.select(test_idx)
    })


def load_iemocap_speaker_independent():
    dataset = load_dataset("AbstractTTS/IEMOCAP", split="train")
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))
    def merge_and_filter(batch):
        emo = batch["major_emotion"]
        if emo == "excited": emo = "happy"
        batch["emotion_4class"] = emo
        return batch
    dataset = dataset.map(merge_and_filter)
    dataset = dataset.filter(lambda b: b["emotion_4class"] in ["angry", "happy", "neutral", "sad"])
    def add_speaker(batch):
        parts = batch["file"].split("_")
        session = parts[0][:5]
        speaker_gender = parts[-1][0]
        batch["speaker_id"] = f"{session}_{speaker_gender}"
        return batch
    dataset = dataset.map(add_speaker)
    splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
    train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=dataset["speaker_id"]))
    return DatasetDict({
        "train": dataset.select(train_idx),
        "test": dataset.select(test_idx)
    })

In [ ]:
LOADERS = {
    "TESS":    load_tess_speaker_independent,
    "RAVDESS": load_ravdess_speaker_independent,
    "CREMA-D": load_cremad_speaker_independent,
    "IEMOCAP": load_iemocap_speaker_independent,
}
datasets_raw = {corpus: loader() for corpus, loader in LOADERS.items()}

###Κλάσεις που αναλύονται

In [ ]:
# Οι 4 κλάσεις για το cross corpus analysis
COMMON_LABELS = ["angry", "happy", "neutral", "sad"]
COMMON_LABEL2ID = {name: i for i, name in enumerate(COMMON_LABELS)}

#Mapping των ετικετών κάθε συνόλου, σε κοινή ετικές ή none αν είναι εκτός
LABEL_MAPS = {
    "TESS": {
        "angry": "angry", "happy": "happy", "neutral": "neutral", "sad": "sad",
        "disgust": None, "fear": None, "pleasant surprise": None,
    },
    "CREMA-D": {
        "anger": "angry",   #διαφορετικό όνομα από τα άλλα σύνολα
        "happy": "happy", "neutral": "neutral", "sad": "sad",
        "disgust": None, "fear": None,
    },
    "RAVDESS": {
        "angry": "angry", "happy": "happy", "neutral": "neutral", "sad": "sad",
        "calm": None, "fearful": None, "disgust": None, "surprised": None,
    },
    "IEMOCAP": {
    "angry": "angry",
    "happy": "happy",
    "neutral": "neutral",
    "sad": "sad",
    }
}

In [ ]:
#Επιστρέφει τους δείκτες εξόδου του μοντέλου που αντιστοιχούν σε μία από τις 4 κοινές κλάσεις.
def get_valid_source_indices(corpus, id2label):
    label_map = LABEL_MAPS[corpus]
    valid = {}
    for idx, raw_label in id2label.items():
        common = label_map.get(raw_label)
        if common is not None:
            valid[idx] = common
    return valid


def remap_prediction(logits_row, corpus, id2label):
    #logits_row: 1D array/tensor, τα raw logits ενός δείγματος από το source model.
    valid_indices = get_valid_source_indices(corpus, id2label)
    restricted = {idx: logits_row[idx] for idx in valid_indices}
    best_idx = max(restricted, key=restricted.get)
    return valid_indices[best_idx]


RAW_LABEL_COLUMN = {
    "TESS": "emotion",
    "CREMA-D": "major_emotion",
    "IEMOCAP": "emotion_4class",
    "RAVDESS": "labels",
}

RAVDESS_LABEL_NAMES = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']


def filter_target_dataset(dataset_split, corpus, raw_label_column):
    label_map = LABEL_MAPS[corpus]

    def get_raw_label_str(example):
        val = example[raw_label_column]
        if corpus == "RAVDESS":
            return RAVDESS_LABEL_NAMES[val]   # int -> string
        return val   # ήδη string στα άλλα corpora

    def has_common_label(example):
        return label_map.get(get_raw_label_str(example)) is not None

    def add_common_label(example):
        raw_str = get_raw_label_str(example)
        example["common_label"] = label_map[raw_str]
        example["common_label_id"] = COMMON_LABEL2ID[example["common_label"]]
        return example

    filtered = dataset_split.filter(has_common_label)
    filtered = filtered.map(add_common_label)
    return filtered

In [ ]:
filtered_test = filter_target_dataset(datasets_raw["IEMOCAP"]["test"], "IEMOCAP", RAW_LABEL_COLUMN["IEMOCAP"])
print(len(filtered_test))
from collections import Counter
print(Counter(filtered_test["common_label"]))

Filter:   0%|          | 0/1414 [00:00<?, ? examples/s]

Map:   0%|          | 0/1414 [00:00<?, ? examples/s]

1414
Counter({'happy': 491, 'neutral': 439, 'sad': 305, 'angry': 179})


In [ ]:
import os
from torch.utils.data import DataLoader

from google.colab import drive
drive.mount('/content/drive')

BATCH_SIZE = 16
RESULTS_DIR = "/content/drive/MyDrive/Colab Notebooks/SER_Thesis/30_cross_corpus/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def make_collate_fn(feature_extractor):
    def collate_fn(batch):
        audios = [ex["audio"]["array"] for ex in batch]
        inputs = feature_extractor(audios, sampling_rate=16000, padding=True, return_tensors="pt")
        common_ids = torch.tensor([ex["common_label_id"] for ex in batch], dtype=torch.long)
        return inputs, common_ids
    return collate_fn


def run_inference_pair(source_corpus, target_corpus, model, feature_extractor, filtered_target_ds):
    id2label = {int(k): v for k, v in model.config.id2label.items()}
    valid_indices = get_valid_source_indices(source_corpus, id2label)  # {model_idx: common_label}

    # ομαδοποίηση δεικτών ανά κοινή κλάση (π.χ. IEMOCAP: exc+hap -> happy)
    indices_per_common = {c: [] for c in COMMON_LABELS}
    for idx, common in valid_indices.items():
        indices_per_common[common].append(idx)

    missing = [c for c, idxs in indices_per_common.items() if len(idxs) == 0]
    if missing:
        raise ValueError(f"{source_corpus}: καμία source κλάση δεν αντιστοιχεί σε {missing}")

    loader = DataLoader(filtered_target_ds, batch_size=BATCH_SIZE, collate_fn=make_collate_fn(feature_extractor))
    device = next(model.parameters()).device

    all_preds, all_gt, all_logits = [], [], []
    with torch.no_grad():
        for inputs, common_ids in loader:
            inputs = {k: v.to(device) for k, v in inputs.items()}
            logits = model(**inputs).logits.cpu().numpy()

            # για κάθε κοινή κλάση, max logit μεταξύ όλων των αντίστοιχων source δεικτών
            restricted = np.stack([
                logits[:, indices_per_common[c]].max(axis=1) for c in COMMON_LABELS
            ], axis=1)   # (batch, 4), σειρά = COMMON_LABELS

            preds = restricted.argmax(axis=1)
            all_preds.append(preds)
            all_gt.append(common_ids.numpy())
            all_logits.append(restricted)

    return {
        "predictions": np.concatenate(all_preds),
        "ground_truth": np.concatenate(all_gt),
        "logits": np.concatenate(all_logits, axis=0),
    }


def save_result(source_corpus, target_corpus, result):
    fname = f"{source_corpus}_to_{target_corpus}.npz"
    path = os.path.join(RESULTS_DIR, fname)
    np.savez(path,
             predictions=result["predictions"],
             ground_truth=result["ground_truth"],
             logits=result["logits"],
             common_labels=np.array(COMMON_LABELS))
    acc = (result["predictions"] == result["ground_truth"]).mean()
    print(f"  {source_corpus} -> {target_corpus}: acc={acc:.3f}  (n={len(result['ground_truth'])})  ✓ αποθηκεύτηκε")


In [ ]:
# --- Προϋπολογισμός των φιλτραρισμένων target test sets (μία φορά, εκτός loop) ---
filtered_targets = {
    corpus: filter_target_dataset(ds["test"], corpus, RAW_LABEL_COLUMN[corpus])
    for corpus, ds in datasets_raw.items()
}

# --- Κύριο 4×4 loop ---
for source_corpus, model in models.items():
    fe = feature_extractors[source_corpus]
    print(f"\n=== Source model: {source_corpus} ===")
    for target_corpus, filtered_ds in filtered_targets.items():
        result = run_inference_pair(source_corpus, target_corpus, model, fe, filtered_ds)
        save_result(source_corpus, target_corpus, result)


=== Source model: TESS ===
  TESS -> TESS: acc=0.566  (n=800)  ✓ αποθηκεύτηκε
  TESS -> RAVDESS: acc=0.286  (n=252)  ✓ αποθηκεύτηκε
  TESS -> CREMA-D: acc=0.266  (n=1022)  ✓ αποθηκεύτηκε
  TESS -> IEMOCAP: acc=0.192  (n=1414)  ✓ αποθηκεύτηκε

=== Source model: RAVDESS ===
  RAVDESS -> TESS: acc=0.260  (n=800)  ✓ αποθηκεύτηκε
  RAVDESS -> RAVDESS: acc=0.619  (n=252)  ✓ αποθηκεύτηκε
  RAVDESS -> CREMA-D: acc=0.523  (n=1022)  ✓ αποθηκεύτηκε
  RAVDESS -> IEMOCAP: acc=0.318  (n=1414)  ✓ αποθηκεύτηκε

=== Source model: CREMA-D ===
  CREMA-D -> TESS: acc=0.339  (n=800)  ✓ αποθηκεύτηκε
  CREMA-D -> RAVDESS: acc=0.587  (n=252)  ✓ αποθηκεύτηκε
  CREMA-D -> CREMA-D: acc=0.787  (n=1022)  ✓ αποθηκεύτηκε
  CREMA-D -> IEMOCAP: acc=0.427  (n=1414)  ✓ αποθηκεύτηκε

=== Source model: IEMOCAP ===
  IEMOCAP -> TESS: acc=0.295  (n=800)  ✓ αποθηκεύτηκε
  IEMOCAP -> RAVDESS: acc=0.306  (n=252)  ✓ αποθηκεύτηκε
  IEMOCAP -> CREMA-D: acc=0.464  (n=1022)  ✓ αποθηκεύτηκε
  IEMOCAP -> IEMOCAP: acc=0.642  (n=1414)

In [ ]:
for corpus, model in models.items():
    id2label = {int(k): v for k, v in model.config.id2label.items()}
    print(f"=== {corpus} ===")
    print("id2label:", id2label)
    valid = get_valid_source_indices(corpus, id2label)
    print("valid (mapped to common labels):", valid)
    print()

=== TESS ===
id2label: {0: 'angry', 1: 'disgust', 2: 'fear', 3: 'happy', 4: 'neutral', 5: 'pleasant surprise', 6: 'sad'}
valid (mapped to common labels): {0: 'angry', 3: 'happy', 4: 'neutral', 6: 'sad'}

=== RAVDESS ===
id2label: {0: 'neutral', 1: 'calm', 2: 'happy', 3: 'sad', 4: 'angry', 5: 'fearful', 6: 'disgust', 7: 'surprised'}
valid (mapped to common labels): {0: 'neutral', 2: 'happy', 3: 'sad', 4: 'angry'}

=== CREMA-D ===
id2label: {0: 'anger', 1: 'disgust', 2: 'fear', 3: 'happy', 4: 'neutral', 5: 'sad'}
valid (mapped to common labels): {0: 'angry', 3: 'happy', 4: 'neutral', 5: 'sad'}

=== IEMOCAP ===
id2label: {0: 'angry', 1: 'happy', 2: 'neutral', 3: 'sad'}
valid (mapped to common labels): {0: 'angry', 1: 'happy', 2: 'neutral', 3: 'sad'}

